# 새 모델(NAD) 실행 노트북 — Moving-Camera AV-SELD

위에서부터 **순서대로** 실행합니다. 바꿀 곳은 **2번 설정 칸**뿐입니다. 런타임은 **GPU**(A100 권장, 고용량 RAM)로 하세요.

**세션이 끊기면**: 1, 2번 셀을 다시 실행하고 → 3, 4번(로컬 준비, 이미 있으면 건너뜀)을 다시 실행한 뒤 → 끊긴 학습 셀을 다시 실행하면 **마지막 에폭 다음부터 이어 합니다**
(에폭 = `steps_per_epoch`번 갱신. 에폭마다 `last.pth`, `train_log.csv`, `train_out.log`를 드라이브의 `runs/<실행 이름>/`에 저장).

| 저장 위치 | 내용 | 크기(예상) |
|---|---|---|
| 코랩 로컬 `/content/nad` | 원본 FOA(train 방), 미리 렌더링 오디오 캐시 | 약 3GB + 약 18GB (세션마다 다시 만듦) |
| 드라이브 `New_Architecture_dev/cache` | 라벨 캐시, CLAP 캐시, OWL-ViT 캐시 | 수십 MB + 약 0.1GB + **약 11~12GB** (7번 전에 다시 확인) |
| 드라이브 `New_Architecture_dev/runs` | 체크포인트, 로그, 예측 | 실행당 약 0.2GB |

`for_dataset/` 안의 파일은 읽기만 합니다(코드는 `__pycache__`도 만들지 않게 `python -B`로 실행).

## 1. 드라이브 연결, 패키지

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

DRIVE_ROOT = "/content/drive/MyDrive/Colab Notebooks/ASR_2026-2"   #@param {type:"string"}
NAD_DIR = f"{DRIVE_ROOT}/New_Architecture_dev"
import os
os.chdir(NAD_DIR)
os.environ['PYTHONDONTWRITEBYTECODE'] = '1'
!pip install -q librosa soundfile pyyaml tqdm
import transformers, huggingface_hub; print("transformers", transformers.__version__, "| huggingface_hub", huggingface_hub.__version__)
!nvidia-smi -L
!free -g | head -2
!df -h /content | tail -1
!df -h /content/drive | tail -1

## 2. 설정 칸
- `seed`: 시드는 1번만 돌립니다.
- `total_steps`: 총 갱신 횟수. **N1-pre와 N1-otf는 같은 값**으로 둡니다(1단계 비교 조건).
- `steps_per_epoch`: 이만큼 갱신할 때마다 val 채점과 체크포인트 저장(이어 하기 단위).
- `stage1_winner`: 6번 비교 뒤, val로 더 나은 1단계 실행 이름을 적습니다(2단계 초기값).

In [ ]:
seed = 1                   #@param {type:"integer"}
batch_size = 32            #@param {type:"integer"}
total_steps = 20000        #@param {type:"integer"}
steps_per_epoch = 500      #@param {type:"integer"}
lr = 0.001                 #@param {type:"number"}
num_workers = 6            #@param {type:"integer"}
train_stages = "0,1"       #@param ["0,1", "0"]
stage1_winner = ""         #@param {type:"string"}
n2_total_steps = 20000     #@param {type:"integer"}
finetune_after_step = 0    #@param {type:"integer"}

def ARGS(preset, **extra):
    sets = dict(drive_root=DRIVE_ROOT, seed=seed, batch_size=batch_size, total_steps=total_steps, steps_per_epoch=steps_per_epoch,
                lr=lr, num_workers=num_workers, train_stages=train_stages)
    sets.update(extra)
    return f'--preset {preset} ' + ' '.join(f'--set {k}="{v}"' for k, v in sets.items())

def run_name(preset):
    import sys; sys.path.insert(0, NAD_DIR)
    from nad.config import load_config
    return load_config(preset, [f'drive_root={DRIVE_ROOT}', f'seed={seed}', f'train_stages={train_stages}'])['run_name']

# 2단계 명령 인자. 이 칸에서 만들므로 stage1_winner 를 바꾸고 이 칸만 다시 실행하면 반영된다.
A2 = ARGS('N2', total_steps=n2_total_steps, init_from=stage1_winner, finetune_after_step=finetune_after_step)
print(ARGS('N1-pre'))
print('2단계 init_from =', repr(stage1_winner))

## 3. 원본 풀기 + [필수 확인] 즉석 변환 = 생성기 출력
STARSS23 `foa_dev.zip`, `metadata_dev.zip`에서 **main20 train 방(4, 6, 7, 12, 13, 14, 21) 녹음만** 로컬에 풉니다(val 방 9, 22와 test 방은 풀지 않고, 불러올 때도 다시 거릅니다). 수 분.
`check_match`: main20 train 클립 3개를 같은 녹음·구간·궤적으로 즉석 변환해 생성기 출력과 비교합니다(오디오 비트 단위, 라벨, 궤적). 모두 `PASS: true`여야 합니다.

In [ ]:
A = ARGS('N1-otf')
!python -B nad_run.py prepare_raw {A}
!python -B nad_run.py check_match {A}

## 4. 미리 렌더링 데이터 캐시
먼저 용량을 봅니다. 오디오 캐시(int16)는 **코랩 로컬**에 만들고(main20 train + S1 약 17GB, val/test 약 1.2GB), 라벨 캐시는 드라이브에 둡니다(작음).
zip에서 wav만 읽으므로 영상은 풀지 않습니다. 처음에는 10~30분(미측정). 끊기면 같은 셀을 다시 실행하면 이어 합니다.

In [ ]:
A = ARGS('N1-pre')
!python -B nad_run.py pre_size {A}
!df -h /content | tail -1

In [ ]:
!python -B nad_run.py prepare_pre {A}

## 5. 확인 (한 번만)
- `check_feat`: GPU 음향 특징(L, R, Mid, Side log-mel, IV, MSC)이 NTU numpy 코드와 같은지(가장자리 제외 최대 차이). 로컬 CPU에서는 log-mel 최대 차이 약 2e-4 dB, IV/MSC 약 1e-5였습니다.
- `oracle`: val 정답을 새 출력 형식으로 바꿔 후처리 → csv → 베이스라인 채점. F가 1에 가까워야 합니다(로컬에서 99.3%; 같은 클래스가 20° 안에 겹치면 합쳐져서 1보다 조금 낮음).

In [ ]:
!python -B nad_run.py check_feat {A}
!python -B nad_run.py oracle {A}

## 6. CLAP 캐시 (고정 LAION-CLAP `laion/clap-htsat-unfused`)
train은 원본 녹음 W(=Mid)에서 녹음 단위로, val/test는 클립 Mid에서 클립 단위로 2초 창을 0.5초씩 뽑습니다. 드라이브 약 0.1GB. 수십 분(미측정).

In [ ]:
!python -B nad_run.py clap {A}

## 7. 1단계 학습: 음향만 (A, B, CMC, E-2, F; 손실은 활성, sinθ, 거리)
두 실행은 **데이터만 다르고 갱신 횟수는 같습니다.**
- `N1-pre`: main20 train + S1 미리 렌더링 데이터
- `N1-otf`: STARSS23 train 방 즉석 변환(매번 새 창, 새 카메라 궤적)

모델 선택 기준은 val `F_basic`입니다(스테레오는 앞/뒤 정보가 없어 1단계는 앞/뒤를 학습하지 않음, 추론 때 모두 앞(+)으로 냄). F_ext도 매 에폭 기록됩니다.
세션이 끊기면 3, 4번을 다시 실행한 뒤 이 셀을 다시 실행하세요(이어 하기).

In [ ]:
!python -B nad_run.py train {ARGS('N1-pre')}
!python -B nad_run.py eval {ARGS('N1-pre')}

In [ ]:
!python -B nad_run.py train {ARGS('N1-otf')}
!python -B nad_run.py eval {ARGS('N1-otf')}

### 1단계 비교 (val)

In [ ]:
import json, pandas as pd
def val_row(name):
    p = f"{NAD_DIR}/runs/{name}/scores_val.json"
    if not os.path.exists(p): return None
    d = json.load(open(p)); s = d['scores']
    return dict(run=name, best_epoch=d.get('best_epoch'), F_ext=s['ext']['F'], F_basic=s['basic']['F'], DOAE_ext=s['ext']['DOA_err'],
                RDE=s['ext']['rel_dist_err'], OSA=s['ext']['onscreen_acc'])
rows = [r for r in (val_row(run_name('N1-pre')), val_row(run_name('N1-otf'))) if r]
pd.DataFrame(rows)

## 8. OWL-ViT 캐시 (고정 `google/owlvit-base-patch32`)
1) **용량을 먼저 봅니다**(드라이브 여유 공간과 비교). 2) 전처리 설정 확인(정사각형 늘리기/여백, 열 방위각). 3) 캐시 만들기: PCA(S0 train 300클립 × 10프레임) → train(+반전), val, test.
클래스 문장은 `configs/owl_prompts.yaml`(바꾸면 새 캐시 폴더). 오래 걸립니다(수 시간 가능, 미측정). 끊기면 다시 실행하면 끝난 묶음은 건너뜁니다.

In [ ]:
!python -B nad_run.py owl_size {A2}
!df -h /content/drive | tail -1

In [ ]:
!python -B nad_run.py owl_check {A2}

In [ ]:
!python -B nad_run.py owl {A2}

## 9. 2단계 학습: 전체 AV 모델 (C, D, E-1 추가, 모든 손실)
`stage1_winner`(2번 설정 칸)의 최고 모델에서 (A), (B), CMC, (E-2), (F)와 특징 통계를 가져오고, (A) 인코더는 고정합니다(`finetune_after_step` > 0이면 그 갱신부터 낮은 학습률로 같이 학습).
2단계는 미리 렌더링 데이터(영상 특징 캐시)로 학습합니다. 그다음 val로 후처리 값(τ, 열 점수 기준, ±창, 보정 켜기/끄기)을 고릅니다.

In [ ]:
assert stage1_winner, '2번 설정 칸에 stage1_winner(1단계 실행 이름)를 적으세요'
assert f'init_from="{stage1_winner}"' in A2, '2번 설정 칸을 다시 실행하세요(A2 에 stage1_winner 가 반영되지 않음)'
!python -B nad_run.py train {A2}
!python -B nad_run.py tune_post {A2}
!python -B nad_run.py eval {A2}

## 10. test 예측과 채점 (최종 보고용)
모델과 후처리는 val로 정했습니다. test는 여기서 **한 번만** 채점합니다(비공개 정답 `private_test.zip`).
비교 대상 B3(S0+S1 9200, `RESULTS_REPORT.md` 11절): test F_ext 12.85 / F_basic 17.30 / DOA_ext 39.2° / onscreen 80.1%.

In [ ]:
runs1 = [('N1-pre', run_name('N1-pre')), ('N1-otf', run_name('N1-otf'))]
for p, nm in runs1:
    if os.path.exists(f"{NAD_DIR}/runs/{nm}/best_model.pth"):
        A1 = ARGS(p)
        !python -B nad_run.py predict {A1}
        !python -B nad_run.py score_test {A1}
if stage1_winner and os.path.exists(f"{NAD_DIR}/runs/{run_name('N2')}/best_model.pth"):
    !python -B nad_run.py predict {A2}
    !python -B nad_run.py score_test {A2}

In [ ]:
def test_row(name):
    p = f"{NAD_DIR}/runs/{name}/scores_test.json"
    if not os.path.exists(p): return None
    s = json.load(open(p))['scores']
    return dict(run=name, F_ext=s['ext']['F'], F_basic=s['basic']['F'], DOAE_ext=s['ext']['DOA_err'], RDE=s['ext']['rel_dist_err'], OSA=s['ext']['onscreen_acc'])
allr = [run_name('N1-pre'), run_name('N1-otf'), run_name('N2')]
ref = dict(run='B3 S0+S1 9200 (reference)', F_ext=0.1285, F_basic=0.1730, DOAE_ext=39.2, RDE=None, OSA=0.801)
pd.DataFrame([r for r in map(test_row, allr) if r] + [ref])